In [1]:
import sys, time, re, json, pathlib
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)

DATA = pathlib.Path("data"); DATA.mkdir(exist_ok=True)
GENES = ["CYP2C19", "CYP2C9", "VKORC1", "DPYD", "TPMT"]
MV = "https://myvariant.info/v1"
AVI_FILE = DATA / "alphagenome_avi.tsv"

ANCHORS = {"rs1799853": "CYP2C9*2", "rs1057910": "CYP2C9*3",
           "rs9923231": "VKORC1 -1639G>A", "rs4244285": "CYP2C19*2",
           "rs12248560": "CYP2C19*17", "rs3918290": "DPYD*2A",
           "rs67376798": "DPYD c.2846A>T", "rs1142345": "TPMT*3C"}

FIELDS = ",".join([
    "dbsnp.rsid",                        # the rs number
    "snpeff.ann.effect",                 # consequence term
    "cadd.phred",                        # CADD
    "dbnsfp.revel.score",                # REVEL
    "dbnsfp.alphamissense",              # AlphaMissense
    "dbnsfp.aa.ref", "dbnsfp.aa.alt", "dbnsfp.aa.pos", "dbnsfp.genename",
    "gnomad_genome.af.af", "gnomad_exome.af.af",
    "clinvar.rcv.clinical_significance",
    "clinvar.rcv.review_status",
    "clinvar.gene.symbol",
    "clinvar.variant_id",                # join key for CADD (see Section 3)
])

def first(x):
    """Some fields come back as a single value, some as a list. Take the first."""
    return (x[0] if x else None) if isinstance(x, list) else x

def dig(d, *path):
    """Walk a nested dict safely. Returns None if any level is missing."""
    for k in path:
        if not isinstance(d, dict):
            return None
        d = d.get(k)
        if d is None:
            return None
    return d

# ClinVar review status, as the familiar star rating
STARS = {"practice guideline": 4,
         "reviewed by expert panel": 3,
         "criteria provided, multiple submitters, no conflicts": 2,
         "criteria provided, conflicting classifications": 1,
         "criteria provided, conflicting interpretations": 1,
         "criteria provided, single submitter": 1,
         "no assertion criteria provided": 0,
         "no classification provided": 0}

# snpEff reports one effect per transcript. Keep the most severe one.
SEVERITY = ["stop_gained", "frameshift_variant", "splice_acceptor_variant",
            "splice_donor_variant", "start_lost", "stop_lost", "missense_variant",
            "splice_region_variant", "synonymous_variant", "5_prime_UTR_variant",
            "3_prime_UTR_variant", "intron_variant", "upstream_gene_variant",
            "downstream_gene_variant", "intergenic_region"]

def all_effects(h):
    """Every effect snpEff reported, across every transcript, split into single terms."""
    ann = dig(h, "snpeff", "ann")
    raw = [a.get("effect") for a in (ann if isinstance(ann, list) else [ann]) if isinstance(a, dict)]
    terms = []
    for e in raw:                       # snpEff emits compound terms joined by "&"
        if e:
            terms += [t for t in e.split("&") if t != "structural_interaction_variant"]
    return sorted(set(terms))

def worst_effect(h):
    """Collapse to ONE term: the most severe across all transcripts."""
    terms = all_effects(h)
    if not terms:
        return None
    return min(terms, key=lambda t: SEVERITY.index(t) if t in SEVERITY else 99)

def to_row(h, gene=None):
    aaref, aaalt = first(dig(h, "dbnsfp", "aa", "ref")), first(dig(h, "dbnsfp", "aa", "alt"))
    aapos = first(dig(h, "dbnsfp", "aa", "pos"))
    rcv = first(dig(h, "clinvar", "rcv")) or {}   # one record per submission; take the first
    rev = rcv.get("review_status")
    am = dig(h, "dbnsfp", "alphamissense", "score")
    return {
        "hgvs_g":        h.get("_id"),
        "rsid":          dig(h, "dbsnp", "rsid"),
        "gene":          first(dig(h, "clinvar", "gene", "symbol")) or first(dig(h, "dbnsfp", "genename")) or gene,
        "clinvar_id":    dig(h, "clinvar", "variant_id"),
        "consequence":   worst_effect(h),
        "consequence_all": ";".join(all_effects(h)),   # keep the full picture
        "aa_change":     f"{aaref}{aapos}{aaalt}" if (aaref and aaalt and aapos) else None,
        "cadd_phred":    dig(h, "cadd", "phred"),
        "revel":         first(dig(h, "dbnsfp", "revel", "score")),
        "alphamissense": first(am) if isinstance(am, list) else am,
        "gnomad_af":     dig(h, "gnomad_genome", "af", "af") or dig(h, "gnomad_exome", "af", "af"),
        "clinvar_sig":   rcv.get("clinical_significance"),
        "clinvar_stars": STARS.get(str(rev).lower(), 0),
    }

FORCE_REFETCH = False

# Known pharmacogenomic alleles, so we can check the fetch worked and
# recognise them later.
ANCHORS = {"rs1799853": "CYP2C9*2", "rs1057910": "CYP2C9*3",
           "rs9923231": "VKORC1 -1639G>A", "rs4244285": "CYP2C19*2",
           "rs12248560": "CYP2C19*17", "rs3918290": "DPYD*2A",
           "rs67376798": "DPYD c.2846A>T", "rs1142345": "TPMT*3C"}

def fetch_gene(gene, verbose=True):
    """Search ClinVar for one gene, annotate every hit, return a tidy DataFrame."""
    r = requests.get(f"{MV}/query", timeout=90, params={
        "q": f"clinvar.gene.symbol:{gene}", "assembly": "hg38",
        "fields": "_id", "size": 1000})
    r.raise_for_status()
    ids = [h["_id"] for h in r.json().get("hits", [])]
    if verbose:
        print(f"  {gene:9s} {len(ids):>5} variants found", end="", flush=True)

    rows = []
    for i in range(0, len(ids), 500):
        rr = requests.post(f"{MV}/variant", timeout=180,
                           data={"ids": ",".join(ids[i:i+500]),
                                 "fields": FIELDS, "assembly": "hg38"})
        rr.raise_for_status()
        rows += [to_row(h, gene) for h in rr.json()]
        time.sleep(0.3)                        # be polite to a free API

    df = pd.DataFrame(rows).drop_duplicates("hgvs_g")

    # MyVariant.info only carries genome-wide CADD on hg19. Ask the same
    # question on hg19 and join on the ClinVar variant ID, which is build-free.
    r19 = requests.get(f"{MV}/query", timeout=90, params={
        "q": f"clinvar.gene.symbol:{gene}", "assembly": "hg19",
        "fields": "clinvar.variant_id,cadd.phred", "size": 1000})
    r19.raise_for_status()
    hits19 = r19.json().get("hits", [])
    cadd = {}
    for h in hits19:
        vid = first(dig(h, "clinvar", "variant_id"))      # occasionally a list
        ph = dig(h, "cadd", "phred")
        if vid is not None and ph is not None:
            cadd[vid] = ph
    df["cadd_phred"] = df.clinvar_id.map(cadd)

    if len(ids) >= 1000 or len(hits19) >= 1000:
        print(f"\n  [!] {gene} hit the 1000-result cap. The two builds may have been")
        print( "      truncated to different sets, so some CADD scores will be missing.")
    if verbose:
        print(f"  ->  {len(df)} annotated")
    return df

def numeric(df):
    """Force the score columns to numbers. Strings here break everything later."""
    for c in ["cadd_phred", "revel", "alphamissense", "spliceai_dg",
              "spliceai_dl", "avi", "gnomad_af", "clinvar_stars"]:
        if c not in df.columns:
            df[c] = np.nan
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def clinvar_label(sig):
    if not isinstance(sig, str): return "none"
    s = sig.lower()
    if "conflict" in s: return "conflicting"
    if "pathogenic" in s and "non-pathogenic" not in s: return "P/LP"
    if "benign" in s: return "B/LB"
    if "uncertain" in s: return "VUS"
    return "other"          # includes "drug response"

STRENGTH = [(18.7, "strong, pathogenic"), (4.3, "moderate, pathogenic"),
            (2.08, "supporting, pathogenic"), (0.48, "no evidence either way"),
            (0.23, "supporting, benign"), (0.053, "moderate, benign"),
            (0.0, "strong, benign")]

def strength_for(lr):
    if lr != lr: return "undefined"
    for cut, name in STRENGTH:
        if lr >= cut: return name
    return "strong, benign"

def calibrate(df, score_col, bands):
    d = df[df.label.isin(["P/LP", "B/LB"]) & df[score_col].notna()].copy()
    d["band"] = pd.cut(d[score_col], bands)
    n_p, n_b = (d.label == "P/LP").sum(), (d.label == "B/LB").sum()
    g = d.groupby("band", observed=False)
    out = pd.DataFrame({"n": g.size(),
                        "n_PLP": g.label.apply(lambda s: (s == "P/LP").sum()),
                        "n_BLB": g.label.apply(lambda s: (s == "B/LB").sum())})
    with np.errstate(divide="ignore", invalid="ignore"):
        out["LR+"] = ((out.n_PLP / max(n_p, 1)) / (out.n_BLB / max(n_b, 1))).round(2)
    out["evidence"] = out["LR+"].map(strength_for)
    out.attrs["n_p"], out.attrs["n_b"] = n_p, n_b
    return out

print("helpers loaded:", ", ".join(["fetch_gene", "numeric", "to_row", "clinvar_label",
                                    "calibrate", "strength_for", "worst_effect"]))

c:\Users\zaina\anaconda3\envs\biot6900\Lib\site-packages\requests\__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


helpers loaded: fetch_gene, numeric, to_row, clinvar_label, calibrate, strength_for, worst_effect


In [2]:
MY_GENE = "CYP2C9"         # not assigned a gene; chose CYP2C9 (disclosed in README)
assert MY_GENE in GENES, f"MY_GENE must be one of {GENES}"
MY_FILE = DATA / f"{MY_GENE}.tsv"
print("working on", MY_GENE)

working on CYP2C9


In [3]:
# Q1 · Fetch (or load from cache), clean, label, quality-check
if MY_FILE.exists() and not FORCE_REFETCH:
    G = pd.read_csv(MY_FILE, sep="\t")
    print(f"loaded {len(G)} rows from cache {MY_FILE}")
else:
    G = fetch_gene(MY_GENE)
    G.to_csv(MY_FILE, sep="\t", index=False)
    print(f"fetched and cached to {MY_FILE}")

G = numeric(G)
G["label"] = G.clinvar_sig.map(clinvar_label)

# Check 1 · count
print(f"\nCHECK 1 · count: {len(G)} variants for {MY_GENE}")
print(G.label.value_counts().to_string())

# Check 2 · anchors (the star alleles that belong to MY gene)
mine = {rs: name for rs, name in ANCHORS.items()
        if name.split()[0].split("*")[0] == MY_GENE}
print(f"\nCHECK 2 · anchors for {MY_GENE}:")
for rs, name in mine.items():
    hit = G[G.rsid == rs]
    if len(hit):
        r = hit.iloc[0]
        print(f"  FOUND   {rs} {name:18s} consequence={r.consequence}  "
              f"gnomAD AF={r.gnomad_af}  ClinVar='{r.clinvar_sig}'")
    else:
        print(f"  MISSING {rs} {name}  -> delete {MY_FILE} and re-fetch before continuing")

# Check 3 · are the score columns populated at all?
print("\nCHECK 3 · score columns populated")
for c in ["cadd_phred", "revel", "alphamissense", "gnomad_af"]:
    n = G[c].notna().sum()
    print(f"  {c:14s} {n:>5} / {len(G)}  ({100*n/len(G):.1f}%)" + ("   <-- EMPTY, re-run fetch" if n == 0 else ""))


loaded 41 rows from cache data\CYP2C9.tsv

CHECK 1 · count: 41 variants for CYP2C9
label
VUS      22
other    10
B/LB      8
P/LP      1

CHECK 2 · anchors for CYP2C9:
  FOUND   rs1799853 CYP2C9*2           consequence=missense_variant  gnomAD AF=0.0881641  ClinVar='drug response'
  FOUND   rs1057910 CYP2C9*3           consequence=missense_variant  gnomAD AF=0.0494537  ClinVar='drug response'

CHECK 3 · score columns populated
  cadd_phred        38 / 41  (92.7%)
  revel             33 / 41  (80.5%)
  alphamissense     33 / 41  (80.5%)
  gnomad_af         29 / 41  (70.7%)


In [4]:
# Q2 · Coverage by consequence class
TOOLS = ["cadd_phred", "revel", "alphamissense"]
if "avi" in G.columns and G["avi"].notna().any():
    TOOLS.append("avi")

cons = G.consequence.fillna("unannotated")
cov = G[TOOLS].notna().groupby(cons).sum()
cov.insert(0, "n_variants", cons.value_counts())
cov = cov.sort_values("n_variants", ascending=False)
print(f"variants scored per tool, by consequence ({MY_GENE})\n")
print(cov.to_string())

print("\noverall % scored by each tool")
print((G[TOOLS].notna().mean() * 100).round(1).to_string())

no_revel = cov.index[cov.revel == 0].tolist()
no_missense_tool = cov.index[(cov.revel == 0) & (cov.alphamissense == 0)].tolist()
no_tool = cov.index[(cov[TOOLS] == 0).all(axis=1)].tolist()
print("\nclasses with ZERO REVEL scores:              ", no_revel)
print("classes with no REVEL AND no AlphaMissense:  ", no_missense_tool)
print("classes NO tool scored at all:               ", no_tool or "none")
print(f"\nvariants with no score from any tool: {G[TOOLS].isna().all(axis=1).sum()}")


variants scored per tool, by consequence (CYP2C9)

                    n_variants  cadd_phred  revel  alphamissense
consequence                                                     
missense_variant            33          33     33             33
synonymous_variant           3           3      0              0
intron_variant               2           2      0              0
unannotated                  2           0      0              0
frameshift_variant           1           0      0              0

overall % scored by each tool
cadd_phred       92.7
revel            80.5
alphamissense    80.5

classes with ZERO REVEL scores:               ['synonymous_variant', 'intron_variant', 'unannotated', 'frameshift_variant']
classes with no REVEL AND no AlphaMissense:   ['synonymous_variant', 'intron_variant', 'unannotated', 'frameshift_variant']
classes NO tool scored at all:                ['unannotated', 'frameshift_variant']

variants with no score from any tool: 3


In [5]:
# Q3 · Frequency filter
# CYP2C9: *2 (~10-12%) and *3 (~6-8%) are common in Europeans, and *3 reaches
# ~10% in parts of South Asia. They reduce warfarin/phenytoin clearance.
# Pharmacogene logic: reduced/increased-function alleles are under little or no
# purifying selection (drug exposure is evolutionarily recent), so frequency is NOT
# evidence against function here. Only drop variants where the ALT allele is the
# majority allele, i.e. the reference genome carries the unusual allele.
MAX_AF = 0.5     # <- justify this in 6.1 (change if your inspection below says so)

def removed_by(df, cut):
    return df[df.gnomad_af > cut]      # NaN AF (never seen in gnomAD) is KEPT

def important(df):
    return df[df.label.isin(["P/LP", "other"])
              | df.clinvar_sig.fillna("").str.contains("drug response", case=False)
              | df.rsid.isin(ANCHORS.keys())]

cols = ["rsid", "consequence", "aa_change", "gnomad_af", "clinvar_sig", "label"]

rem = removed_by(G, MAX_AF)
print(f"MAX_AF = {MAX_AF}: removes {len(rem)} of {len(G)} variants")
imp = important(rem).assign(anchor=lambda x: x.rsid.map(ANCHORS))
print(f"  P/LP, drug-response or anchor variants removed: {len(imp)}")
print(imp[cols + ["anchor"]].to_string(index=False) if len(imp) else "  (none)")

rem_rare = removed_by(G, 0.001)
imp_rare = important(rem_rare).assign(anchor=lambda x: x.rsid.map(ANCHORS))
print(f"\nCONTRAST · a 0.001 rare-disease filter would remove {len(rem_rare)} variants,")
print(f"  of which {len(imp_rare)} are P/LP, drug-response or anchor alleles:")
print(imp_rare.sort_values("gnomad_af", ascending=False)[cols + ["anchor"]].to_string(index=False)
      if len(imp_rare) else "  (none)")

G_unfiltered = G.copy()
G = G[G.gnomad_af.isna() | (G.gnomad_af <= MAX_AF)].copy()
print(f"\nG after filter: {len(G)} variants  (kept {G.gnomad_af.isna().sum()} with no gnomAD AF)")


MAX_AF = 0.5: removes 1 of 41 variants
  P/LP, drug-response or anchor variants removed: 1
     rsid    consequence aa_change  gnomad_af clinvar_sig label anchor
rs1856908 intron_variant       NaN   0.645943 association other    NaN



CONTRAST · a 0.001 rare-disease filter would remove 10 variants,
  of which 6 are P/LP, drug-response or anchor alleles:
      rsid      consequence aa_change  gnomad_af   clinvar_sig label   anchor
 rs1856908   intron_variant       NaN   0.645943   association other      NaN
 rs9332220   intron_variant       NaN   0.190775   association other      NaN
 rs1799853 missense_variant     R144C   0.088164 drug response other CYP2C9*2
 rs1057910 missense_variant     I359L   0.049454 drug response other CYP2C9*3
 rs7900194 missense_variant     R150H   0.015809 drug response other      NaN
rs28371685 missense_variant     R335W   0.007015 drug response other      NaN

G after filter: 40 variants  (kept 12 with no gnomAD AF)


In [6]:
# Q4 · Calibrate a predictor into evidence
SCORE_COL = "cadd_phred"                 # scores every consequence class (REVEL is missense-only)
BANDS     = [0, 10, 20, 25, 30, 100]     # top edge 100 so CADD > 50 (e.g. stop_gained) is not dropped
MIN_N     = 10
POOL_WITH = None    # small-gene route: e.g. "CYP2C19" to pool a second gene. Disclose it if you use it.

C = G
if POOL_WITH:
    f2 = DATA / f"{POOL_WITH}.tsv"
    G2 = pd.read_csv(f2, sep="\t") if f2.exists() else fetch_gene(POOL_WITH)
    if not f2.exists(): G2.to_csv(f2, sep="\t", index=False)
    G2 = numeric(G2); G2["label"] = G2.clinvar_sig.map(clinvar_label)
    G2 = G2[G2.gnomad_af.isna() | (G2.gnomad_af <= MAX_AF)]
    C = pd.concat([G, G2], ignore_index=True)
    print(f"POOLED {MY_GENE} + {POOL_WITH} for calibration")

cal = calibrate(C, SCORE_COL, BANDS)
n_p, n_b = cal.attrs["n_p"], cal.attrs["n_b"]
print(cal.to_string())
print(f"\ncalibration rests on {n_p + n_b} labelled variants: {n_p} P/LP and {n_b} B/LB")
print(f"('drug response' alleles are labelled 'other' and are NOT in this calibration: "
      f"{(C.label == 'other').sum()} of them)")

small = cal[cal.n < MIN_N]
print(f"\nbands resting on fewer than {MIN_N} variants: {len(small)} of {len(cal)}")
for b, r in small.iterrows():
    print(f"  {b}: n={r.n} (P/LP {r.n_PLP}, B/LB {r.n_BLB}), LR+={r['LR+']}  -> not usable as evidence")
zero = cal[(cal.n_PLP == 0) | (cal.n_BLB == 0)]
if len(zero):
    print(f"bands with a zero count (LR is 0 or infinite, i.e. undefined, not 'strong'): {list(zero.index.astype(str))}")

if min(n_p, n_b) < MIN_N:
    print(f"\n[!] Only {n_p} P/LP and {n_b} B/LB. Too few to calibrate {MY_GENE} credibly.")
    print("    Take a route: set POOL_WITH to a second gene, or state calibration is not possible.")


           n  n_PLP  n_BLB   LR+                evidence
band                                                    
(0, 10]    4      1      3  2.33  supporting, pathogenic
(10, 20]   1      0      1  0.00          strong, benign
(20, 25]   3      0      3  0.00          strong, benign
(25, 30]   0      0      0   NaN               undefined
(30, 100]  0      0      0   NaN               undefined

calibration rests on 8 labelled variants: 1 P/LP and 7 B/LB
('drug response' alleles are labelled 'other' and are NOT in this calibration: 9 of them)

bands resting on fewer than 10 variants: 5 of 5
  (0, 10]: n=4 (P/LP 1, B/LB 3), LR+=2.33  -> not usable as evidence
  (10, 20]: n=1 (P/LP 0, B/LB 1), LR+=0.0  -> not usable as evidence
  (20, 25]: n=3 (P/LP 0, B/LB 3), LR+=0.0  -> not usable as evidence
  (25, 30]: n=0 (P/LP 0, B/LB 0), LR+=nan  -> not usable as evidence
  (30, 100]: n=0 (P/LP 0, B/LB 0), LR+=nan  -> not usable as evidence
bands with a zero count (LR is 0 or infinite, i.e. unde

In [7]:
# Q5 · Classify VUS against the calibrated bands, then export
# A band only counts as evidence if it is backed by >= MIN_N variants and both counts are non-zero.
usable = (cal.n >= MIN_N) & (cal.n_PLP > 0) & (cal.n_BLB > 0)
band_evidence = {b: (e if ok else "no evidence (band unreliable)")
                 for b, e, ok in zip(cal.index, cal.evidence, usable)}

V = G[G.label == "VUS"].copy()
V["band"] = pd.cut(V[SCORE_COL], BANDS)
V["evidence"] = V["band"].astype(object).map(band_evidence)
V.loc[V[SCORE_COL].isna(), "evidence"] = "not scored (no information)"
V["evidence"] = V["evidence"].fillna("not scored (no information)")

def call(ev):
    if "pathogenic" in ev: return f"VUS, leaning pathogenic (PP3 {ev.split(',')[0]})"
    if "benign" in ev:     return f"VUS, leaning benign (BP4 {ev.split(',')[0]})"
    return "VUS, unchanged"
# A predictor alone never makes a variant LP or LB under ACMG; it only adds one line of evidence.
V["provisional_call"] = V.evidence.map(call)
V["direction"] = V.provisional_call.str.extract(r"leaning (\w+)")[0].fillna("stays put")

print(f"{len(V)} VUS in {MY_GENE}\n")
print(V.direction.value_counts().to_string())
print("\nby evidence:")
print(V.evidence.value_counts().to_string())

out_cols = ["rsid", "gene", "consequence", "aa_change", SCORE_COL, "gnomad_af",
            "evidence", "provisional_call"]
out = V[out_cols].assign(gene=MY_GENE)
OUT = f"pgx_classified_{MY_GENE}.csv"
out.to_csv(OUT, index=False)
print(f"\nwrote {OUT}: {len(out)} rows")
print(out.head(10).to_string(index=False))


22 VUS in CYP2C9

direction
stays put    22

by evidence:
evidence
no evidence (band unreliable)    22

wrote pgx_classified_CYP2C9.csv: 22 rows
        rsid   gene      consequence aa_change  cadd_phred  gnomad_af                      evidence provisional_call
         NaN CYP2C9 missense_variant      C10Y       3.849        NaN no evidence (band unreliable)   VUS, unchanged
rs1210894054 CYP2C9 missense_variant     D468H      22.900   0.000004 no evidence (band unreliable)   VUS, unchanged
         NaN CYP2C9 missense_variant     E300V      26.500        NaN no evidence (band unreliable)   VUS, unchanged
rs2031306544 CYP2C9 missense_variant      L15I       9.711        NaN no evidence (band unreliable)   VUS, unchanged
 rs370407644 CYP2C9 missense_variant     E274Q      23.300   0.000007 no evidence (band unreliable)   VUS, unchanged
         NaN CYP2C9 missense_variant      P35S      25.300        NaN no evidence (band unreliable)   VUS, unchanged
rs1226372739 CYP2C9 missense_variant

In [8]:
SKIP_ATLAS   = True
MAX_VARIANTS = 200

import os
ATLAS_STATUS = "not run"

def to_variant_string(hgvs):
    """Atlas wants 1-based GRCh38 chr:pos:ref>alt. It does NOT accept rsIDs."""
    m = re.match(r"(chr[\w]+):g\.(\d+)([ACGT])>([ACGT])", str(hgvs))
    return f"{m.group(1)}:{m.group(2)}:{m.group(3)}>{m.group(4)}" if m else None

if SKIP_ATLAS:
    print("SKIPPED. Disclose this in your report and answer Q6.2 from reasoning.")
    ATLAS_STATUS = "skipped"
else:
    try:
        from alphagenome.atlas import atlas
        from alphagenome.data import genome
        key = os.environ.get("ALPHAGENOME_API_KEY") or os.environ.get("ALPHA_GENOME_API_KEY")
        if not key:
            raise RuntimeError(
                "No API key in your environment. Did you relaunch Jupyter from a NEW\n"
                "  terminal after setting it? Check with:\n"
                "    import os; print(os.environ.get('ALPHAGENOME_API_KEY'))\n"
                "  Or set SKIP_ATLAS = True.")
        client = atlas.create(key, timeout=30)

        pri = G.assign(_nc=~G.consequence.fillna("").str.contains("missense")) \
               .sort_values("_nc", ascending=False).head(MAX_VARIANTS)
        rows, errs = [], 0
        for n, (_, r) in enumerate(pri.iterrows(), 1):
            vs = to_variant_string(r.hgvs_g)
            if not vs:
                continue
            try:
                res = client.query_variant(
                    genome.Variant.from_str(vs),
                    requested_scorers=["AVI_SCORE", "AVI_SCORE_FEATURE_IMPORTANCE"])
                ad = res["AVI_SCORE"]
                cdf = float(ad.layers["quantiles"][0, 0])
                fi = res["AVI_SCORE_FEATURE_IMPORTANCE"]
                rows.append({"rsid": r.rsid,
                             "avi_phred": round(-10*np.log10(max(1-cdf, 1e-12)), 2),
                             "top_feature": fi.var["name"].values[int(np.argmax(np.abs(fi.X[0])))]})
            except Exception:
                errs += 1
            if n % 50 == 0:
                print(f"  {n}/{len(pri)} queried", flush=True)

        avi = pd.DataFrame(rows)
        if len(avi):
            avi.to_csv(DATA / f"avi_{MY_GENE}.tsv", sep="\t", index=False)
            G = G.merge(avi, on="rsid", how="left")
            G["avi"] = G.avi_phred
            ATLAS_STATUS = "queried live"
            print(f"\nscored {len(avi)} variants ({errs} failures)")
    except ImportError:
        print('alphagenome not installed. pip install "alphagenome>=0.9.0", then restart.')
        print("Or set SKIP_ATLAS = True.")
    except Exception as e:
        print(f"{type(e).__name__}: {e}")

if "avi" not in G.columns: G["avi"] = np.nan
if "top_feature" not in G.columns: G["top_feature"] = np.nan
G["avi"] = pd.to_numeric(G["avi"], errors="coerce")

if G.avi.notna().sum() > 0:
    a = G[G.avi.notna()]
    cols = [c for c in ["rsid", "consequence", "avi_phred", "top_feature",
                        "revel", "cadd_phred", "label"] if c in a.columns]
    print(f"\nhighest-impact {MY_GENE} variants by AVI\n")
    print(a.sort_values("avi", ascending=False)[cols].head(10).to_string(index=False))
    print("\nwhat is driving the scores (read this BEFORE the numbers)")
    print(a.top_feature.value_counts().head(8).to_string())
else:
    nc = G[~G.consequence.fillna("").str.contains("missense")]
    print(f"\nnon-missense variants in {MY_GENE}, which is what the Atlas is for: {len(nc)}")
    print(nc.consequence.value_counts().to_string())

print(f"\n[Atlas status: {ATLAS_STATUS}]")

SKIPPED. Disclose this in your report and answer Q6.2 from reasoning.

non-missense variants in CYP2C9, which is what the Atlas is for: 7
consequence
synonymous_variant    3
frameshift_variant    1
intron_variant        1

[Atlas status: skipped]


In [9]:
d = G.copy()
coding = d.consequence.isin(["missense_variant", "synonymous_variant"])

d["flag_cadd_vs_revel"]  = (d.cadd_phred > 25) & (d.revel < 0.35)
d["flag_revel_vs_am"]    = (d.revel - d.alphamissense).abs() > 0.4
d["flag_scored_but_vus"] = (d.label == "VUS") & (d.revel > 0.7)
d["flag_common_but_plp"] = (d.gnomad_af > 0.01) & (d.label.isin(["P/LP", "other"]))
d["flag_no_tool_at_all"] = d[["revel", "alphamissense", "cadd_phred"]].isna().all(axis=1)
if d.avi.notna().any():
    d["flag_atlas_only"] = (d.avi > 20) & d.revel.isna() & d.alphamissense.isna()

flags = [c for c in d.columns if c.startswith("flag_")]
print(f"discordant variants in {MY_GENE}\n")
print(d[flags].sum().to_string())

hits = d[d[flags].any(axis=1)]
cols = [c for c in ["rsid", "note", "consequence", "consequence_all", "aa_change",
                    "cadd_phred", "revel", "alphamissense", "avi", "gnomad_af",
                    "label"] if c in d.columns]
print(f"\n{len(hits)} variants trip at least one rule\n")
print(hits[cols].head(15).to_string(index=False) if len(hits) else "  (none)")

if len(hits) == 0:
    print("\nNone. That is a legitimate result. Say so in your report and explain why")
    print("this gene's variants are concordant, rather than loosening the rules until")
    print("something fires.")

discordant variants in CYP2C9

flag_cadd_vs_revel     4
flag_revel_vs_am       2
flag_scored_but_vus    0
flag_common_but_plp    4
flag_no_tool_at_all    3

12 variants trip at least one rule

        rsid        consequence                          consequence_all aa_change  cadd_phred  revel  alphamissense  avi  gnomad_af label
         NaN frameshift_variant frameshift_variant;splice_region_variant       NaN         NaN    NaN            NaN  NaN   0.003103  B/LB
  rs28371685   missense_variant                         missense_variant     R335W      27.500  0.222         0.2408  NaN   0.007015 other
         NaN   missense_variant                         missense_variant     E300V      26.500  0.320         0.6387  NaN        NaN   VUS
   rs1057910   missense_variant                         missense_variant     I359L      20.400  0.070         0.1487  NaN   0.049454 other
         NaN                NaN                                      NaN       NaN         NaN    NaN           

Extra checks on unscored and P/LP variants

In [10]:
show = ["hgvs_g", "clinvar_id", "consequence", "consequence_all", "aa_change",
        "cadd_phred", "gnomad_af", "clinvar_sig"]
print("NO TOOL SCORED THESE:")
print(G_unfiltered[G_unfiltered[["cadd_phred","revel","alphamissense"]].isna().all(axis=1)][show].to_string(index=False))
print("\nTHE ONE P/LP VARIANT:")
print(G_unfiltered[G_unfiltered.label == "P/LP"][show].to_string(index=False))

NO TOOL SCORED THESE:
             hgvs_g  clinvar_id        consequence                          consequence_all aa_change  cadd_phred  gnomad_af                 clinvar_sig
chr10:g.94949282del      285601 frameshift_variant frameshift_variant;splice_region_variant       NaN         NaN   0.003103 Benign/Likely benign; other
chr10:g.94942290C>C      163021                NaN                                      NaN       NaN         NaN        NaN               drug response
chr10:g.94981296A>A      177709                NaN                                      NaN       NaN         NaN        NaN               drug response

THE ONE P/LP VARIANT:
             hgvs_g  clinvar_id      consequence  consequence_all aa_change  cadd_phred  gnomad_af clinvar_sig
chr10:g.94947919T>G        8410 missense_variant missense_variant     L208V       0.002        NaN  Pathogenic


In [11]:
   print(G_unfiltered[G_unfiltered.rsid.isin(["rs1799853", "rs1057910"])][["rsid", "hgvs_g"]])
   print(G_unfiltered[G_unfiltered.clinvar_id == 8410][["aa_change", "cadd_phred", "revel", "alphamissense"]])

         rsid               hgvs_g
7   rs1057910  chr10:g.94981296A>C
37  rs1799853  chr10:g.94942290C>T
   aa_change  cadd_phred  revel  alphamissense
13     L208V       0.002  0.118         0.1381
